# Query Web IQ through MCP

Connect directly to the Web IQ MCP endpoint, inspect the tools it exposes, and call them to search the web, search the news, and read a web page. Web IQ authenticates with an API key that you send in the `x-apikey` header.

## Step 1: Load configuration and create an MCP session helper

Load the Web IQ key from the project `.env` file. If `WEB_IQ_KEY` is missing, get the key for your Web IQ account from the Web IQ portal and add it to `.env`.

In [ ]:
import os
from contextlib import asynccontextmanager

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]
WEB_IQ_MCP_URL = "https://api.microsoft.ai/v3/mcp"


@asynccontextmanager
async def open_webiq_session():
    async with httpx.AsyncClient(
        headers={"x-apikey": WEB_IQ_KEY},
        timeout=httpx.Timeout(30, read=120),
    ) as http_client:
        async with streamable_http_client(WEB_IQ_MCP_URL, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Web IQ configuration loaded")

## Step 2: List the available tools

Open an MCP session and inspect every tool that Web IQ exposes. Each tool covers a different kind of web content: web pages, news, images, videos, blended search (`sonic`), and reading a single page (`browse`).

In [ ]:
from rich.console import Console

console = Console()

async with open_webiq_session() as session:
    tools_response = await session.list_tools()

for tool in tools_response.tools:
    console.print(f"[bold cyan]{tool.name}[/]: {tool.description}")
    console.print(f"  Parameters: {', '.join(tool.inputSchema.get('properties', {}))}")

## Step 3: Search the web

Call the `web` tool with a query. Each result has a title, URL, and the page content, which Web IQ returns with lightweight markup tags. The helper below strips those tags and shows a short snippet of each result. Change `query` to try other searches.

In [ ]:
import re


def show_results(results, snippet_length=300):
    for result in results:
        console.print(f"[bold]{result['title']}[/]")
        console.print(f"[link={result['url']}]{result['url']}[/link]")
        text = re.sub(r"<[^>]+>", " ", result.get("content", ""))
        text = re.sub(r"\s+", " ", text).strip()
        console.print(text[:snippet_length] + "...\n")


query = "mental health benefits trends for employers"

async with open_webiq_session() as session:
    result = await session.call_tool("web", {"query": query, "maxResults": 5})

show_results(result.structuredContent["webResults"])

## Step 4: Search the news

The `news` tool returns recent news articles. Use it for questions about current events, where freshness matters more than general relevance.

In [ ]:
query = "home improvement retail trends"

async with open_webiq_session() as session:
    result = await session.call_tool("news", {"query": query, "maxResults": 5})

show_results(result.structuredContent["newsResults"])

## Step 5: Read a web page

The `browse` tool fetches a single page and returns its content. Agents use it to read a page in full after finding it in search results.

In [ ]:
url = "https://learn.microsoft.com/azure/search/agentic-retrieval-overview"

async with open_webiq_session() as session:
    result = await session.call_tool("browse", {"url": url, "maxLength": 2000})

page = result.structuredContent
show_results([page], snippet_length=1500)

## Next steps

You called Web IQ directly through MCP. In the Foundry IQ exercise, `foundryiq-webiq.ipynb` adds the same MCP endpoint as a knowledge source, so a knowledge base can combine web results with your own documents.